# Spatial Disparity Analysis: SPI, Accessibility, LISA, and Spatial Regression

This notebook continues the project after preprocessing and tehsil-level SPI/AI creation.

It uses the latest processed tehsil file reflected in the current workflow:
- `data/processed/tehsil_spi_ai_fullstudy.geojson`

If that GeoJSON does not expose the final snow-inclusive `spi` field directly,
the helper code automatically merges the processed `spi` values from the CSV exports by `shapeID`.

Main outputs:
- Global Moran's I for SPI, Accessibility, and the scenic-access gap
- LISA cluster maps
- priority tehsil identification for high-scenic / low-access areas
- OLS plus spatial-model diagnostics
- spatial lag/error model selection
- sensitivity analysis for alternative SPI weights


In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

from scripts.spatial_analysis_utils import get_analysis_paths, run_full_analysis

PROJECT_ROOT = Path('.').resolve()
PATHS = get_analysis_paths(PROJECT_ROOT)
PATHS


## Run Full Analysis

This cell computes the remaining analysis and saves tables, GeoJSON, and plots into:

- `outputs/spatial_disparity_analysis/`


In [ ]:
results = run_full_analysis(PROJECT_ROOT, save_results=True)

gdf = results['gdf']
global_moran = results['global_moran']
priority_tehsils = results['priority_tehsils']
regression = results['regression']
sensitivity = results['sensitivity']
saved_paths = results['saved_paths']

print('Rows analysed:', len(gdf))
print('Priority tehsils:', len(priority_tehsils))
print('Selected spatial model:', regression['selected_model'])
saved_paths


## Global Moran's I


In [ ]:
global_moran


## Priority High-Scenic / Low-Access Tehsils


In [ ]:
priority_tehsils[
    [
        'shapeName',
        'spi',
        'ai',
        'spi_z',
        'ai_z',
        'gap_index',
        'gap_cluster',
    ]
].head(20)


## LISA Cluster Counts


In [ ]:
pd.DataFrame({
    'SPI LISA': gdf['spi_cluster'].value_counts(),
    'AI LISA': gdf['ai_cluster'].value_counts(),
    'Gap LISA': gdf['gap_cluster'].value_counts(),
}).fillna(0).astype(int)


## OLS and Spatial Diagnostics


In [ ]:
regression['ols_coefficients']


In [ ]:
regression['lm_tests']


In [ ]:
pd.DataFrame([regression['residual_moran']])


In [ ]:
regression['spatial_coefficients']


## SPI Source Check


In [ ]:
gdf[['shapeName', 'spi', 'spi_source']].head(10)


## Sensitivity Analysis


In [ ]:
sensitivity


## Generated Plots


In [ ]:
for key, path in saved_paths.items():
    print(f'{key}: {path}')


In [ ]:
from IPython.display import Image, display

for key in ['choropleths', 'lisa_maps', 'regression_plot', 'sensitivity_plot']:
    print(key)
    display(Image(filename=str(saved_paths[key])))
